# OmniVoice TTS demo (Colab)

Generates speech with [k2-fsa/OmniVoice](https://huggingface.co/k2-fsa/OmniVoice). Shares one Google Drive cache folder with the other NLP 2026 audio demos (model weights + recorded example clips), so you don't re-download a few GB every time you open any of these notebooks.

**Note:** if you share one Drive folder with many workshop attendees, Google Drive's anti-abuse download quota can start throttling it under heavy simultaneous use. This notebook falls back to a normal Hugging Face download if the Drive cache doesn't work.

In [ ]:
# Colab already ships a working torch/CUDA pair, so only install what's missing.
# datasets: one public sample clip for cloning. pydub: decode the browser mic
# recording (needs Colab's preinstalled ffmpeg).
!pip install -q omnivoice soundfile datasets pydub

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Shared cache for all NLP 2026 audio demos (OmniVoice, Whisper, Canary):
# models/ holds downloaded weights, recordings/ holds saved + pre-cached
# example clips, so every demo notebook reuses the same downloads.
# For a shared workshop copy: share this folder from your own Drive, have
# attendees "Add shortcut to Drive", then point this at that shortcut's path.
DRIVE_CACHE_DIR = "/content/drive/MyDrive/nlp2026_audio_cache"
RECORDINGS_DIR = f"{DRIVE_CACHE_DIR}/recordings"
os.makedirs(RECORDINGS_DIR, exist_ok=True)
os.environ["HF_HOME"] = f"{DRIVE_CACHE_DIR}/models"

In [ ]:
import torch
from omnivoice import OmniVoice

MODEL_ID = "k2-fsa/OmniVoice"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32

try:
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
except Exception as e:
    print(f"Drive cache unavailable ({e!r}); falling back to a fresh download.")
    os.environ.pop("HF_HOME", None)
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)

In [ ]:
# instruct: comma-separated, at most one item per category (English only)
#   gender: male, female
#   age: child, teenager, young adult, middle-aged, elderly
#   pitch: very low pitch, low pitch, moderate pitch, high pitch,
#     very high pitch
#   style: whisper
#   accent: american accent, british accent, australian accent, canadian accent,
#     indian accent, japanese accent, korean accent, portuguese accent,
#     russian accent, chinese accent
# e.g. "female, elderly, low pitch, british accent"
EXAMPLES = [
    {
        "name": "en_female_welcome",
        "text": "Hello NLP Summer School of 2026. Welcome in Kinit.",
        "instruct": "female",
    },
    {
        "name": "sk_male_welcome",
        "text": "Ahoj, letná škola NLP 2026. Vitajte v Kinite.",
        "instruct": "male",
        "language": "sk",
    },
    {
        "name": "sk_male_welcome_02",
        "text": "Ahoj, letná škola NLP 2026. Vitajte v Kinyte. ",
        "instruct": "male",
        "language": "sk",
    },
]

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

os.makedirs("outputs", exist_ok=True)

def synthesize(path, **kwargs):
    audio = model.generate(**kwargs)
    if isinstance(audio, list):
        audio = audio[0]
    sf.write(path, audio, 24000)
    display(Audio(path))

for example in EXAMPLES:
    print(f"Generating '{example['name']}'...")
    synthesize(
        f"outputs/{example['name']}.wav",
        text=example["text"],
        instruct=example["instruct"],
        language=example.get("language"),
    )

## Voice Cloning

Instead of describing a voice with `instruct`, clone one from a short reference clip + its transcript (`ref_text` is optional — omit it and the model auto-transcribes via Whisper).

In [ ]:
from datasets import load_dataset

sample = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")[0]
ref_audio = sample["audio"]["array"]
ref_sr = sample["audio"]["sampling_rate"]
ref_text = sample["text"]

print(f"Reference text: {ref_text}")
display(Audio(ref_audio, rate=ref_sr))

In [ ]:
cloned_text = "Now I can speak using someone else's voice, thanks to voice cloning!"
synthesize("outputs/cloned_sample_voice.wav", text=cloned_text, ref_audio=(ref_audio, ref_sr), ref_text=ref_text)

### Record & clone your own voice

Run the setup cell once, then run the record cell below it — allow microphone access when prompted, click **Start** when you're ready to read the sentence, and it stops automatically after a few seconds. Re-run just the record cell for retakes; the recording is saved to the shared Drive cache folder so it survives runtime resets.

To play back a pre-cached example instead (e.g. an edge case or failure mode) rather than recording live, comment out the `record()` line and uncomment `load_recording(...)` with a filename from `recordings/`.

In [ ]:
from google.colab.output import eval_js
from IPython.display import Javascript
from base64 import b64decode
from pydub import AudioSegment
import io
import json
import numpy as np

RECORD_SECONDS = 8
PROMPT_SENTENCE = "The quick brown fox jumps over the lazy dog. NLP summer school is going to be great!"

_RECORD_JS = """
async function record(ms, sentence) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

  const div = document.createElement('div');
  div.innerHTML = `<p>Click, then read: <b>"${sentence}"</b> (auto-stops after ${ms / 1000}s)</p><button>🎤 Start</button>`;
  document.body.appendChild(div);
  const btn = div.querySelector('button');
  await new Promise(resolve => { btn.onclick = resolve; });
  btn.textContent = `● Recording... (${ms / 1000}s)`;

  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  recorder.start();
  setTimeout(() => recorder.stop(), ms);
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  div.remove();

  const reader = new FileReader();
  reader.readAsDataURL(new Blob(chunks));
  return new Promise(resolve => { reader.onloadend = () => resolve(reader.result); });
}
"""

def _to_waveform(segment):
    segment = segment.set_channels(1)
    waveform = np.array(segment.get_array_of_samples()).astype(np.float32)
    waveform /= 1 << (8 * segment.sample_width - 1)
    return waveform, segment.frame_rate

def record(seconds=RECORD_SECONDS, sentence=PROMPT_SENTENCE):
    display(Javascript(_RECORD_JS))
    data_url = eval_js(f"record({seconds * 1000}, {json.dumps(sentence)})")
    raw = b64decode(data_url.split(",", 1)[1])
    return _to_waveform(AudioSegment.from_file(io.BytesIO(raw)))

def load_recording(name):
    """Load a cached example clip (from recordings/) instead of recording live."""
    return _to_waveform(AudioSegment.from_file(f"{RECORDINGS_DIR}/{name}"))

my_voice_path = f"{RECORDINGS_DIR}/my_voice.wav"

In [ ]:
print(f"Allow microphone access, then click Start when ready — recording stops automatically after {RECORD_SECONDS}s.")
my_voice, my_voice_sr = record()
# my_voice, my_voice_sr = load_recording("example_name.wav")

sf.write(my_voice_path, my_voice, my_voice_sr)
print(f"Saved recording to {my_voice_path}")
display(Audio(my_voice, rate=my_voice_sr))

In [ ]:
my_cloned_text = "Surprise, this is my voice, cloned by an AI, saying something I never actually said!"
synthesize("outputs/my_voice_clone.wav", text=my_cloned_text, ref_audio=(my_voice, my_voice_sr), ref_text=PROMPT_SENTENCE)